In [ ]:
import pandas as pd
import fastf1

In [ ]:
# years 24,25,26 complete
years = [2026]
sessions = ['Session1', 'Session2', 'Session3', 'Session4', 'Session5'] 
df = pd.read_csv('./events/finished.csv')   
df = df[df.columns.to_list()[1:]].reset_index(drop=True)

unsuccessfull_year = []
unsuccessfull_track = []
for year in years:
    year_df = df[df['Year'] == year]
    for race in year_df['EventName'].to_list():
        if year == 2026 and race not in [
            'Spanish Grand Prix','Italian Grand Prix'
        ]:
            continue

        race_df = year_df[year_df['EventName'] == race]
        race_name = race
        for session in sessions:
            session = race_df[session].item()
            if session in ['Race', 'Sprint', 'Qualifying', 'Sprint Qualifying', 'Sprint Shootout']:
                try:
                    session_data = fastf1.get_session(year, race, session)
                    session_data.load()

                    session_type = session_data.session_info['Type']
                    name = f'{year}_{race}_{session}'
                    results = session_data.results
                    laps = session_data.laps
                    weather = session_data.weather_data

                    # exporting dfs to parquets

                    results.to_parquet(f'./data/{name}_results.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                    laps.to_parquet(f'./data/{name}_laps.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                    weather.to_parquet(f'./data/{name}_weather.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')

                except:
                    unsuccessfull_year.append(year)
                    unsuccessfull_track.append(race)

u_df = pd.DataFrame({'Year': unsuccessfull_year, 'Track': unsuccessfull_track})
clean_dfs = []
u_df.to_csv('Failed Raw.csv')

for year in u_df['Year'].unique():
    df = u_df.loc[u_df['Year'] == year]
    df.drop_duplicates(subset='Track', inplace=True)
    clean_dfs.append(df)

try:
    fail_df = pd.concat(clean_dfs).reset_index(drop=True)
except:
    fail_df = clean_dfs.reset_index(drop=True)


fail_df.to_csv('Failed.csv')
